# 📊 Speech Enhancement — Step 4: Evaluate Performance

This notebook measures how well the model denoises speech using three standard metrics:

| Metric | What it measures | Range |
|--------|-----------------|-------|
| **SNR** | Signal-to-noise ratio improvement | Higher = better (dB) |
| **PESQ** | Perceptual speech quality (ITU standard) | –0.5 → 4.5, higher = better |
| **STOI** | Speech intelligibility | 0 → 1, higher = better |

### Prerequisites
- Notebooks 01, 02, and 03 completed.
- Drive contains `data/sounds/` QC pairs (clean + noisy) from `create_data()`.
- Drive contains `predictions/` denoised outputs from notebook 03.

In [30]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted ✓


In [31]:
# ── 2. Install dependencies ────────────────────────────────────────────────
%pip install -q librosa soundfile pesq pystoi

In [32]:
# ── 3. Clone / pull repo & add src/ to path ────────────────────────────────
import subprocess, sys, os

REPO = 'https://github.com/theweird-kid/speech_enhancement.git'  # <── update
REPO_DIR = '/content/speech_enhancement_alt'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('Repo ready ✓')

Repo ready ✓


In [33]:
# ── 4. Config & imports ────────────────────────────────────────────────────
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import pandas as pd
from pesq import pesq
from pystoi import stoi
import config as C

SR = C.SAMPLE_RATE   # 8000 Hz
print(f'Sample rate: {SR} Hz')
print(f'Sounds dir : {C.SOUND_DIR}')
print(f'Pred dir   : {C.PRED_DIR}')

Sample rate: 8000 Hz
Sounds dir : /content/drive/MyDrive/speech_enhancement/data/sounds
Pred dir   : /content/drive/MyDrive/speech_enhancement/predictions


In [34]:
# ── 5. Helper functions ────────────────────────────────────────────────────

def load_wav(path, sr=SR):
    """Load a WAV and return a float32 array resampled to `sr`."""
    y, _ = librosa.load(path, sr=sr)
    return y.astype(np.float32)

def align(a, b):
    """Trim both arrays to the same length."""
    n = min(len(a), len(b))
    return a[:n], b[:n]

def compute_snr(clean, enhanced):
    noise = clean - enhanced
    return 10 * np.log10(np.sum(clean**2) / (np.sum(noise**2) + 1e-8))

def compute_metrics(clean, test, sr=SR):
    """Return dict of SNR, PESQ, STOI for a (clean, test) pair."""
    c, t = align(clean, test)
    return {
        'SNR (dB)': round(compute_snr(c, t), 3),
        'PESQ'    : round(pesq(sr, c, t, 'nb'), 3),
        'STOI'    : round(stoi(c, t, sr, extended=False), 3),
    }

print('Helpers defined ✓')

Helpers defined ✓


In [ ]:
# ── 6. Load QC audio saved by create_data() ────────────────────────────────
# prepare_data.py saves three concatenated WAVs in sounds/:
#   clean_voice_long.wav  — all clean frames joined
#   noisy_voice_long.wav  — matching noisy frames joined

sounds_dir = C.SOUND_DIR

# List what's actually in the sounds dir so you can see the real filenames
print("Files in sounds/:", os.listdir(sounds_dir))

CLEAN_WAV = os.path.join(sounds_dir, 'clean_voice_long.wav')
NOISY_WAV = os.path.join(sounds_dir, 'noisy_voice_long.wav')

clean_a = load_wav(CLEAN_WAV)
noisy_a = load_wav(NOISY_WAV)

print(f'\nClean audio : {len(clean_a)/SR:.1f} s')
print(f'Noisy audio : {len(noisy_a)/SR:.1f} s')

print('\n=== Noisy vs Clean (before enhancement) ===')
before = compute_metrics(clean_a, noisy_a)
for k, v in before.items():
    print(f'  {k}: {v}')


Found 0 QC sample pairs
Empty DataFrame
Columns: []
Index: []


In [ ]:
# ── 7. Load denoised & compute after-metrics ───────────────────────────────
# Run notebook 03 on noisy_voice_long.wav first, save output as:
DENOISED_WAV = os.path.join(C.PRED_DIR, 'denoised_noisy_voice_long.wav')

if os.path.exists(DENOISED_WAV):
    denoised_a = load_wav(DENOISED_WAV)
    print(f'Denoised audio: {len(denoised_a)/SR:.1f} s')

    print('\n=== Denoised vs Clean (after enhancement) ===')
    after = compute_metrics(clean_a, denoised_a)
    for k, v in after.items():
        print(f'  {k}: {v}')

    # Summary table
    import pandas as pd
    metrics = ['SNR (dB)', 'PESQ', 'STOI']
    df = pd.DataFrame({
        'Metric':      metrics,
        'Before':      [before[m] for m in metrics],
        'After':       [after[m]  for m in metrics],
    })
    df['Improvement'] = (df['After'] - df['Before']).round(3)
    print('\n', df.to_string(index=False))
else:
    print(f'[INFO] Not found: {DENOISED_WAV}')
    print('→ Run notebook 03 on noisy_voice_long.wav to generate the denoised file.')



=== Average metrics across all samples ===
Empty DataFrame
Columns: []
Index: [mean]


In [37]:
# ── 8. Bar chart: before vs after for each metric ─────────────────────────
metrics = ['SNR (dB)', 'PESQ', 'STOI']

before_means = [df_results[f'{m} (before)'].mean() for m in metrics]
after_means  = [df_results[f'{m} (after)'].mean()  for m in metrics
                if f'{m} (after)' in df_results.columns]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, before_means, width, label='Noisy (before)', color='#e05c5c')
bars2 = ax.bar(x + width/2, after_means,  width, label='Denoised (after)', color='#4caf7d')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Speech Enhancement — Before vs After', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.bar_label(bars1, fmt='%.3f', padding=3)
ax.bar_label(bars2, fmt='%.3f', padding=3)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.savefig(os.path.join(C.WEIGHTS_DIR, 'performance_chart.png'), dpi=120)
plt.show()

KeyError: 'SNR (dB) (before)'

In [ ]:
# ── 9. Per-sample PESQ scatter plot ───────────────────────────────────────
if 'PESQ (after)' in df_results.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    idx = df_results['sample'].astype(str)
    ax.plot(idx, df_results['PESQ (before)'], 'o--', color='#e05c5c', label='Noisy')
    ax.plot(idx, df_results['PESQ (after)'],  's-',  color='#4caf7d', label='Denoised')
    ax.set_xlabel('Sample index')
    ax.set_ylabel('PESQ score')
    ax.set_title('PESQ per sample — Noisy vs Denoised')
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

In [ ]:
# ── 10. Quick test on a single custom pair ─────────────────────────────────
# Change these paths to compare any clean/noisy/denoised WAV you have

CLEAN_WAV   = os.path.join(C.SOUND_DIR, 'clean_voice_sample_0.wav')   # ← update if needed
NOISY_WAV   = os.path.join(C.SOUND_DIR, 'noisy_voice_sample_0.wav')   # ← update if needed
DENOISED_WAV = os.path.join(C.PRED_DIR, 'denoised_noisy_voice_sample_0.wav')  # ← update if needed

clean_a   = load_wav(CLEAN_WAV)
noisy_a   = load_wav(NOISY_WAV)
denoised_a = load_wav(DENOISED_WAV)

print('=== Before enhancement ===')
for k, v in compute_metrics(clean_a, noisy_a).items():
    print(f'  {k}: {v}')

print('\n=== After enhancement ===')
for k, v in compute_metrics(clean_a, denoised_a).items():
    print(f'  {k}: {v}')

In [ ]:
# ── 11. Spectrogram: clean / noisy / denoised side-by-side ────────────────
import librosa.display

def to_spec(audio, n_fft=C.N_FFT, hop=C.HOP_LENGTH_FFT):
    S = librosa.stft(audio, n_fft=n_fft, hop_length=hop)
    return librosa.amplitude_to_db(np.abs(S), ref=np.max)

specs  = [to_spec(clean_a), to_spec(noisy_a), to_spec(denoised_a)]
titles = ['Clean (reference)', 'Noisy (input)', 'Denoised (output)']

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, S, title in zip(axes, specs, titles):
    librosa.display.specshow(S, sr=SR, hop_length=C.HOP_LENGTH_FFT,
                             x_axis='time', y_axis='hz', ax=ax, cmap='magma')
    ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle('Spectrograms: Clean / Noisy / Denoised', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(C.WEIGHTS_DIR, 'spectrogram_comparison.png'), dpi=120)
plt.show()

In [ ]:
# ── 12. Playback ───────────────────────────────────────────────────────────
import IPython.display as ipd

for label, audio in [('🔇 Clean (reference)', clean_a),
                     ('🔊 Noisy (input)',      noisy_a),
                     ('✨ Denoised (output)',  denoised_a)]:
    print(label)
    display(ipd.Audio(audio, rate=SR))